# Evals and Observability

**What you will build:** a small **evaluation suite** that scores your agent on a set of cases, plus one-line **observability**. This is new versus the n8n course — and it's what separates a demo from something you can trust in production. It follows directly from 0.0's hard truth: LLM output is non-deterministic, so you measure quality, you don't eyeball it.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/16_evals_and_observability.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0" "pydantic-evals>=2.0,<3.0"   # verified against pydantic-ai 2.2 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## Why evals

You changed a prompt and it "seems better." Better on what? On how many cases? An eval turns that gut feeling into a number you can track as you change models and prompts. The workflow: define **cases** (input + what you expect), pick **evaluators** (how to score), run the **dataset** against your agent.

## A minimal eval suite

We'll evaluate a small classification agent.

In [ ]:
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import EqualsExpected

classifier = Agent(model, system_prompt="Classify the message as exactly one word: 'spam' or 'ham'.")

async def classify(text: str) -> str:
    r = await classifier.run(text)
    return r.output.strip().lower()

dataset = Dataset(
    name="spam-detector",
    cases=[
        Case(name="promo",  inputs="WIN a FREE iPhone now!!! click here", expected_output="spam"),
        Case(name="normal", inputs="Are we still on for lunch tomorrow?", expected_output="ham"),
        Case(name="invoice", inputs="Your invoice for March is attached.", expected_output="ham"),
    ],
    evaluators=[EqualsExpected()],   # compares the output against each case's expected_output
)

report = await dataset.evaluate(classify)
report.print(include_input=True, include_output=True)

That table is your ground truth. Change the model in the setup cell, rerun this one cell, and you can *see* whether the cheaper model still passes — instead of hoping. This is how you compare models responsibly.

## Error analysis: the loop that makes evals a tool, not a ritual

A score is only useful if you act on it. The real workflow is a loop: run the eval, look at what **failed**, find the cause, fix it, re-run. Let's add a case the current classifier gets wrong.

In [ ]:
hard = Dataset(
    name="hard-cases",
    cases=[Case(name="newsletter", inputs="Your weekly product newsletter -- 3 new features inside.", expected_output="ham")],
    evaluators=[EqualsExpected()],
)
report = await hard.evaluate(classify)
report.print(include_input=True, include_output=True)   # likely FAILS: 'newsletter' looks promotional -> 'spam'

The failure is informative: the model treats anything promotional as spam, but a newsletter the user subscribed to is *ham*. The eval didn't just hand us a number — it pointed at a missing distinction in the prompt. Fix the cause (the system prompt), then re-run the same case.

In [ ]:
classifier2 = Agent(model, system_prompt=(
    "Classify the message as 'spam' (unsolicited bulk or junk) or 'ham' (wanted mail, "
    "INCLUDING newsletters and updates the user signed up for). Reply with exactly one word."))

async def classify2(text: str) -> str:
    r = await classifier2.run(text)
    return r.output.strip().lower()

report = await hard.evaluate(classify2)
report.print(include_input=True, include_output=True)   # now PASSES

That's error-driven development: the failing case pointed at the fix, and the eval confirmed it worked. Do this in a loop — add a case every time the agent surprises you — and your eval set becomes a growing spec of what "working" means. This loop, not the score itself, is what separates evals-as-engineering from evals-as-ceremony.

## Custom evaluator: LLM as judge

Built-in evaluators (`Contains`, `EqualsExpected`, `IsInstance`) cover objective checks. For subjective quality — "is the tone empathetic?" — you write your own by subclassing `Evaluator`. Here's one that uses an LLM as the judge: the same idea as the `judge()` function in notebook 0.5, now wired into the test harness so it runs over a whole dataset.

In [ ]:
from pydantic_evals.evaluators import Evaluator, EvaluatorContext

judge_agent = Agent(model, system_prompt="You are a strict, fair reviewer.")

class ToneJudge(Evaluator[str]):
    """Use an LLM to judge whether the output sounds professional and empathetic."""
    async def evaluate(self, ctx: EvaluatorContext[str]) -> float:
        verdict = await judge_agent.run(
            f"Does this response sound professional and empathetic? Answer 'yes' or 'no'.\n\n"
            f"Response: {ctx.output}")
        return 1.0 if "yes" in verdict.output.lower() else 0.0

# Evaluate a support-reply agent with the LLM judge:
reply_agent = Agent(model, system_prompt="You write brief, professional customer-support replies.")

async def write_reply(complaint: str) -> str:
    r = await reply_agent.run(complaint)
    return r.output

tone_dataset = Dataset(
    name="tone-eval",
    cases=[Case(name="angry", inputs="Your app deleted my data and I am furious!")],
    evaluators=[ToneJudge()],
)
report = await tone_dataset.evaluate(write_reply)
report.print(include_input=True, include_output=True)

```{note}
This is the same code-rule vs LLM-judge hierarchy from notebook 0.5, now applied to automated testing. Prefer code checks when the rule is objective; reach for an LLM judge only for subjective quality. (pydantic-evals also ships a built-in `LLMJudge(rubric=...)` — we hand-rolled one here to show the `Evaluator` mechanism, but reach for the built-in in real use.) See the [pydantic-evals docs](https://pydantic.dev/docs/ai/evals/getting-started/quick-start/).
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a case that fails.** Insert a spam message labelled `"ham"` into the first dataset and watch the report catch it.
2. **Write a length evaluator.** Create a custom `Evaluator` that scores `1.0` when the output is under 20 characters.
3. **Compare models.** Run the same dataset with two different `MODEL_NAME` values and compare pass rates.
::::

## Observability in one line

When an agent misbehaves you need to see every model call, tool call, and token. **Logfire** (from the Pydantic team) instruments PydanticAI with one call. It's optional and needs a free account, so this cell is illustrative — uncomment to use it.

In [ ]:
# !pip install -q "pydantic-ai-slim[logfire]"
# import logfire
# logfire.configure()               # prompts for a free token on first run
# logfire.instrument_pydantic_ai()  # now every agent.run(...) is traced
print("Observability is optional. Uncomment above to trace runs in Logfire.")

```{note}
In n8n you watched execution on the canvas. In code, observability is something you instrument — Logfire here, or LangSmith in the LangGraph world. Same purpose: see what the agent actually did.
```

**What's next:** the milestone of Block 1 — **1.7**, the **knowledge agent**: an agent that answers from *your* documents (RAG).